# Crime Analytics Rio Grande do Sul, Brazil

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

import warnings 
warnings.filterwarnings('ignore')

In [ ]:
# Set random seed for reproducibility
seed = 42

## 1. Exploratory Data Analysis

In [ ]:
# Defining functions for EDA
def display_feature_list(features, feature_type):
    '''
    This function displays the features within each list for each type of data
    '''

    print(f"\n{feature_type} Features: ")
    print(', '.join(features) if features else 'None')


def describe_df(df):
    """
    This function prints some basic info on the dataset and 
    sets global variables for feature lists.
    """

    global categorical_features, continuous_features, binary_features
    categorical_features = [
        col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])
    ]
    
    binary_features = [col for col in df.columns if df[col].nunique(
    ) <= 2 and df[col].dtype != 'object']
    continuous_features = [
        col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])
    ]

    print(f"\n{type(df).__name__} shape: {df.shape}")
    print(f"\n{df.shape[0]:,.0f} samples")
    print(f"\n{df.shape[1]:,.0f} attributes")
    display_feature_list(categorical_features, 'Categorical')
    display_feature_list(continuous_features, 'Continuous')
    display_feature_list(binary_features, 'Binary (0 or 1)')
    print(f'\nData Types: \n{df.dtypes}')
    print(
        f'\nMissing Data Percentage: \n{(df.isnull().sum() / len(df) * 100).round(2).sort_values().astype(str) + "%"}')
    print(f'\nDuplicates: {df.duplicated().sum()}')
    print(
        f'\nCategorical Feature Cardinality: \n{df[categorical_features].nunique().sort_values(ascending=False)}')
    print(
        f'\nFeatures with Zero Variance: {", ".join([col for col in df.columns if df[col].nunique() == 1]) if any(df[col].nunique() == 1 for col in df.columns) else "None"}')

    negative_valued_features = [
        col for col in df[continuous_features] if (df[col] < 0).any()]
    print(
        f'\nFeatures with Negative Values: {", ".join(negative_valued_features) if negative_valued_features else "None"}')

    print('\nStatistical Summary: \n')
    display(df.describe().T)
    print(f'\n{type(df).__name__} Head: \n')
    display(df.head(5))
    print(f'\n{type(df).__name__} Tail: \n')
    display(df.tail(5))

In [ ]:
df = pd.read_csv('../datasets/raw/crime_df_sample.csv')
describe_df(df)

**1. General Structure**
- The dataset sample consists of **19,998 rows** and **13 columns**.
- There are **no duplicate rows**.

**2. Data Types & Required Conversions**
- **Temporal Features:** The `data_fato` and `hora_fato` columns are currently `object` (string) types. These must be converted to `datetime` to enable time-series analysis, feature extraction (e.g., day of the week, month, hour of the day), and proper sorting.
- **Identifier:** The `sequencia` column appears to be a unique identifier. It has high variance and is likely **not** useful as a predictive feature for a machine learning model. It can probably be dropped during the feature selection phase.

**3. Missing Data**
- we have 3 categorical features with a considerable amount of missing data:
    - **`bairro`**: 19.44% missing
    - **`sexo_vitima`**: 13.84% missing
    - **`cor_vitima`**: 13.76% missing

- A strategy for handling these nulls will be essential, such as imputation with the mode, creating an 'Unknown' category, or using more advanced imputation techniques. We will have to analyze further.

**4. Feature Cardinality and Variance**
- **Zero-Variance Feature:** The `tipo_fato` column has only **1 unique value**, meaning it has zero variance. It provides no information and should be **removed** from the dataset.
- **High-Cardinality Features:** Several features have a very large number of unique values (`hora_fato`, `bairro`, `data_fato`). This will make them difficult to use directly in models that require one-hot encoding, for example. Feature engineering will be critical here (e.g., extracting the hour from `hora_fato`, or grouping `bairro` into larger regions).
- **Low-Cardinality Features:** `grupo_fato`, `local_fato`, `sexo_vitima`, and `cor_vitima` have a small number of unique values and are good candidates for standard categorical encoding.

**5. Statistical Summary Insights**
- **`quantidade_vitimas`**: The number of victims is heavily skewed. The mean is close to 1 (0.92), and the 75th percentile is also 1, indicating that the vast majority of incidents involve a single victim.
- **`idade_vitima`**: The age of victims ranges from 0 to 98, with a mean of ~35. The minimum value of **0** is worth investigating—it could represent an infant or be a placeholder for missing/unknown data.

**Summary & Next Steps**
The dataset is a good starting point but requires considerable preprocessing. The immediate next steps should be:
1.  **Clean:** Drop the `tipo_fato` (zero variance) and `sequencia` (identifier) columns.
2.  **Convert:** Transform `data_fato` and `hora_fato` to datetime objects.
3.  **Impute:** Define and apply a strategy to handle missing values in `bairro`, `sexo_vitima`, and `cor_vitima`.
4.  **Feature Engineer:** Create new, more useful features from high-cardinality and datetime columns.
5.  **Investigate:** Explore the records where `idade_vitima` is 0 to understand their meaning.

### 1.2 Cleaning and Transforming Data

In [ ]:
# Creating a copy of the original dataframe 
df_eda = df.copy()

# Dropping `tipo_fato` and `sequencia` cols
df_eda.drop(columns=['tipo_fato', 'sequencia'], inplace=True)

# Creating a `timestamp` feature with `data_fato` and `hora_fato`
df_eda['data_fato'] = pd.to_datetime(df_eda['data_fato'])
time_delta = pd.to_timedelta(df['hora_fato'].astype(str), errors='coerce')
df_eda['timestamp'] = df_eda['data_fato'] + time_delta
df_eda.drop(columns=['data_fato', 'hora_fato'], inplace=True)

print("After transformations:")
print(df_eda.dtypes)
print('\n' + '---'*50)
df_eda.head(5)

In [ ]:
# Translating features from Brazilian Portuguese to English
feature_map = {
    'grupo_fato': 'crime_group',
    'tipo_enquadramento': 'legal_classification',
    'municipio_fato': 'municipality',
    'local_fato': 'location_type',
    'bairro': 'neighborhood',
    'quantidade_vitimas': 'victim_count',
    'idade_vitima': 'victim_age',
    'sexo_vitima': 'victim_gender',
    'cor_vitima': 'victim_race',
}
df_eda.rename(columns=feature_map, inplace=True)
print("Column names after PT -> EN translation: \n")
print(df_eda.columns.to_list())

In [ ]:
# Checking unique values for each feature
features_list = df_eda.columns.to_list()
features_list.remove('timestamp')
print("\nUnique values for each feature:\n")

for feat in features_list:
    print(f"\nFeature: {feat}\n")
    print(df_eda[feat].unique())

**Brazilian Portuguese to English Translation**
- To improve readability, we will translate values into English as well
- The values in the column `legal_classification`, however, must be preserved in their original Portuguese as they represent **specific articles and definitions in the Brazilian Penal Code**. Direct translation could lead to a loss of legal nuance.
    - For context, here are translations of some classifications in the dataset:
        - **LESAO CORPORAL**: Bodily Injury
        - **ESTELIONATO**: Fraud
        - **OUTROS FURTOS**: Other Thefts
        - **ROUBO A PEDESTRE**: Robbery of a Pedestrian
        - **AMEACA**: Threat

In [ ]:
# Translating values from Brazilian Portuguese to English
# Note: `legal_classification` values must remain in Portuguese due to legal context
# (specific articles and definitions in the Brazilian Penal Code)
crime_group_map = {
    'CRIMES': 'Crimes',
    'CONTRAVENCOES': 'Misdemeanors'
    }

location_type_map = {
    'OUTROS': 'Others',
    'VIA PUBLICA': 'Public Street',
    'RESIDENCIA': 'Residence',
    'ESTABELECIMENTO COMERCIAL': 'Commercial Establishment',
    'ESTABELECIMENTO DIVERSAO': 'Entertainment Venue',
    'HOSPITAIS/CLINICAS': 'Hospitals/Clinics',
    'ETABELECIMENTO ENSINO': 'Educational Institution',
    'INTERIOR COLETIVOS': 'Inside Public Transport',
    'METRO/RODOVIARIA': 'Subway/Bus Station'
}

victim_gender_map = {
    'Feminino': 'Female',
    'Masculino': 'Male',
    'Sem Especificação': 'Not Specified'
}

victim_race_map = {
    'Parda': 'Brown',
    'Branca': 'White',
    'Preta': 'Black',
    'Amarela': 'Yellow',
    'Indígena': 'Indigenous',
    'Sem informação': 'Not Informed',
    'Sem Especificação': 'Not Specified'
}

# Applying the mappings
df_eda['crime_group'] = df_eda['crime_group'].map(crime_group_map)
df_eda['location_type'] = df_eda['location_type'].map(location_type_map)
df_eda['victim_gender'] = df_eda['victim_gender'].map(victim_gender_map)
df_eda['victim_race'] = df_eda['victim_race'].map(victim_race_map)

translated_features = ['crime_group', 'location_type', 'victim_gender', 'victim_race']

print("Unique values after PT -> EN translation (except legal_classification):\n")
for feat in translated_features:
    print(f"\nFeature: {feat}\n")
    print(df_eda[feat].unique())

### 1.3 Data Visualization

In [ ]:
# --- Temporal Analysis ---
df_eda['year'] = pd.to_datetime(df_eda['timestamp']).dt.year
df_eda['month'] = pd.to_datetime(df_eda['timestamp']).dt.month
df_eda['day_of_week'] = pd.to_datetime(df_eda['timestamp']).dt.day_name()
df_eda['hour'] = pd.to_datetime(df_eda['timestamp']).dt.hour

crimes_by_year = df_eda['year'].value_counts().sort_index()
crimes_by_month = df_eda['month'].value_counts().sort_index()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
crimes_by_day = df_eda['day_of_week'].value_counts().reindex(day_order)
crimes_by_hour = df_eda['hour'].value_counts().sort_index()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Crimes by Year', 'Crimes by Month', 'Crimes by Day of Week', 'Crimes by Hour of Day')
)

fig.add_trace(
    go.Bar(x=crimes_by_year.index, y=crimes_by_year.values),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=crimes_by_month.index, y=crimes_by_month.values),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=crimes_by_day.index, y=crimes_by_day.values),
    row=2, col=1
)
fig.add_trace(
    go.Bar(x=crimes_by_hour.index, y=crimes_by_hour.values),
    row=2, col=2
)

fig.update_layout(
    title_text="Temporal Crime Analysis",
    height=800,
    showlegend=False
)

fig.update_yaxes(title_text="Number of Crimes")
fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_xaxes(title_text="Month", row=1, col=2)
fig.update_xaxes(title_text="Day of Week", row=2, col=1)
fig.update_xaxes(title_text="Hour of Day", row=2, col=2)



fig.show()


print(f"\nDate range: {pd.to_datetime(df_eda['data_fato']).min()} to {pd.to_datetime(df_eda['data_fato']).max()}")
print(f"Most common day: {df_eda['day_of_week'].mode()[0]}")


if not df_eda['hour'].dropna().empty:
    print(f"Peak hour: {int(df_eda['hour'].mode()[0])}:00")
